# 10 — Domain Models: SecureBERT vs. vanilla BERT

**Goal:** Swap out `bert-base-uncased` for a **cybersecurity-domain** pretrained model and see whether the domain adaptation pays off on CTI tasks.

## What you'll learn

1. CTI-domain BERT options and where they come from
2. Why domain pretraining matters (vocabulary and representation)
3. A direct A/B on NER (easier to see the effect than classification)
4. When the extra effort is and isn't worth it

## Step 1 — Domain BERT landscape

| Model | HF ID | Notes |
|---|---|---|
| **SecureBERT** | `ehsanaghaei/SecureBERT` | Trained on cybersecurity reports, CVEs, blog posts |
| **CyBERT** | `markusbayer/CySecBERT` | Cybersecurity-focused, smaller |
| **SecBERT** | `jackaduma/SecBERT` | Similar goals, alternative pretraining corpus |
| **CyBERT (RoBERTa)** | `priyank-m/CyBERT` | RoBERTa base, security corpus |

SecureBERT is the most cited starting point. It's built on RoBERTa, so the tokenizer is slightly different from BERT (BPE vs WordPiece), but the workflow is almost identical.

## Step 2 — Compare tokenization of CTI terms

The clearest place domain pretraining shows up is the **tokenizer vocabulary**. If `APT28` is a single token in SecureBERT but fragments to `apt` + `##28` in vanilla BERT, the domain model has a head start.

In [1]:
from transformers import AutoTokenizer

vanilla = AutoTokenizer.from_pretrained("bert-base-uncased")
secure = AutoTokenizer.from_pretrained("ehsanaghaei/SecureBERT")

terms = [
    "APT28", "Emotet", "CVE-2021-44228", "Log4Shell", "Mimikatz",
    "Cobalt Strike", "ransomware", "phishing", "exfiltration",
]

print(f"{'Term':<20} {'BERT tokens':<40} {'SecureBERT tokens'}")
print("-" * 90)
for t in terms:
    v = vanilla.tokenize(t)
    s = secure.tokenize(t)
    print(f"{t:<20} {str(v):<40} {s}")

Term                 BERT tokens                              SecureBERT tokens
------------------------------------------------------------------------------------------
APT28                ['apt', '##28']                          ['APT', '28']
Emotet               ['em', '##ote', '##t']                   ['Em', 'otet']
CVE-2021-44228       ['cv', '##e', '-', '2021', '-', '44', '##22', '##8'] ['CVE', '-', '2021', '-', '44', '228']
Log4Shell            ['log', '##4', '##sh', '##ell']          ['Log', '4', 'Shell']
Mimikatz             ['mimi', '##kat', '##z']                 ['M', 'im', 'ikatz']
Cobalt Strike        ['cobalt', 'strike']                     ['C', 'ob', 'alt', 'ĠStrike']
ransomware           ['ransom', '##ware']                     ['ransomware']
phishing             ['phi', '##shing']                       ['phishing']
exfiltration         ['ex', '##filtration']                   ['ex', 'filtration']


**Expected:** SecureBERT produces fewer fragments for common security terms. Fewer fragments → shorter sequences, stronger representations for the domain, better label alignment.

## Step 3 — A/B NER fine-tuning

Reuse the tiny NER dataset from notebook 5. Train both models with identical hyperparameters and compare F1. We skip the full training here and just sketch the swap — you've run the loop before in notebook 5.

In [2]:
# All you change from notebook 5 is the checkpoint string:
#   model_checkpoint = "ehsanaghaei/SecureBERT"
# Everything else (tokenize_and_align, TrainingArguments, Trainer) stays the same,
# BECAUSE AutoTokenizer and AutoModelForTokenClassification handle the RoBERTa-vs-BERT difference.

# Minimal driver (commented out to avoid accidentally re-running a long job):
# from transformers import AutoModelForTokenClassification
# model = AutoModelForTokenClassification.from_pretrained(
#     "ehsanaghaei/SecureBERT", num_labels=len(label_list),
#     id2label=id2label, label2id=label2id,
# )
# ...same Trainer setup, call trainer.train()
print("Swap only requires changing the model_checkpoint string.")

Swap only requires changing the model_checkpoint string.


## Step 4 — What to expect quantitatively

On published CTI NER benchmarks (DNRTI, CASIE), SecureBERT typically beats vanilla BERT by **1–3 F1 points**. Gains are biggest for:
- Rare threat-actor and malware names (better tokenization helps)
- Short training datasets (the domain pretraining compensates for data scarcity)

Gains are smaller when:
- Your dataset is large (the backbone matters less)
- Your entity types are very generic (e.g., `ORG`)

## Step 5 — Practical watch-outs

- **Size**: SecureBERT is RoBERTa-base (~125M params), similar to BERT-base. Memory and speed are comparable.
- **Tokenizer edge cases**: RoBERTa's tokenizer treats leading spaces differently. When using `is_split_into_words=True`, behavior is correct, but if you pass raw strings carelessly you may see `Ġ`-prefixed tokens.
- **License**: check each model's license before shipping in a commercial product.

## Your turn — exercises

1. Re-run notebook 5 with `ehsanaghaei/SecureBERT` and record F1. Compare to the vanilla-BERT baseline.
2. Re-run notebook 8 (classification) with SecureBERT. Does domain pretraining help classification as much as NER?
3. Count the **average tokens per sentence** in the notebook 5 training data with each tokenizer. How much shorter are SecureBERT's sequences? (Less padding → faster training.)

## Next notebook

**`11_inference_pipeline.ipynb`** — tie everything together. One function: `analyze_report(text)` → `(entities, category)`. This is what you'd deploy.